# Fraud Dataset: Create Kuzu Tables and Relationships

This notebook is a beginner lab for the lowest-level Kuzu steps. Instead of calling the repository loader, you will manually create the fraud node tables, relationship tables, nodes, and relationships.

Use this notebook before the full GraphRAG demo if students need to understand how the graph database is physically created.

## Learning outcome

By the end, students should be able to explain and run these steps:

1. Create a Kuzu database folder.
2. Create node tables with primary keys.
3. Create relationship tables with `FROM` and `TO` node labels.
4. Insert nodes from `data/fraud.json`.
5. Create relationships by matching source and target primary keys.
6. Query and visualize the resulting fraud graph.

## 0. Setup

From the repository root, install the demo dependencies and open this notebook:

```bash
python -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\activate
python -m pip install --upgrade pip
pip install -r requirements-demo.txt
python -m jupyter lab notebooks/fraud_kuzu_tables_relationships_demo.ipynb
```

In [ ]:
from pathlib import Path
import shutil
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

OUTPUT_ROOT = PROJECT_ROOT / 'output'
MANUAL_DB = OUTPUT_ROOT / 'fraud_manual_schema.kuzu'

PROJECT_ROOT

In [ ]:
from IPython.display import HTML, Markdown, SVG, display

from agentic_knowledge_graph.db import execute, import_kuzu
from agentic_knowledge_graph.domains import get_domain
from agentic_knowledge_graph.loader import cypher_literal, load_dataset, property_map
from agentic_knowledge_graph.visualization import fraud_graph_svg, records_to_html, table_to_html

kuzu = import_kuzu()
spec = get_domain('fraud')
dataset = load_dataset('fraud')

display(Markdown(f'Using Kuzu Python package from `{kuzu.__file__}`'))

## 1. Understand the fraud schema

A Kuzu knowledge graph has two schema layers:

- **Node tables** store entities such as fraud types and detection methods.
- **Relationship tables** store allowed directed edges between node tables.

For this dataset, `name` is the primary key for every node table. That makes the notebook easy to read: relationships can say `from: Machine Learning` and `to: Payment Fraud`.

In [ ]:
display(Markdown('### Node tables from the project schema'))
display(HTML(records_to_html([
    {
        'Node table': node.name,
        'Primary key': node.primary_key,
        'Columns': ', '.join(f'{name}: {kind}' for name, kind in node.columns),
    }
    for node in spec.nodes
], ('Node table', 'Primary key', 'Columns'))))

display(Markdown('### Relationship tables from the project schema'))
display(HTML(records_to_html([
    {
        'Relationship': rel.name,
        'From': rel.from_node,
        'To': rel.to_node,
        'Properties': ', '.join(f'{name}: {kind}' for name, kind in rel.columns),
    }
    for rel in spec.relationships
], ('Relationship', 'From', 'To', 'Properties'))))

## 2. Write the Kuzu `CREATE NODE TABLE` statements

A node table declares the label, properties, and primary key. In Kuzu, the primary key uniquely identifies a node in that table.

For example, this tells Kuzu that each `DetectionMethod` is uniquely identified by `name`:

```cypher
CREATE NODE TABLE DetectionMethod(
    name STRING,
    method_type STRING,
    description STRING,
    PRIMARY KEY(name)
)
```

In [ ]:
node_table_ddl = [
    '''CREATE NODE TABLE FraudType(
        name STRING,
        severity STRING,
        description STRING,
        PRIMARY KEY(name)
    )''',
    '''CREATE NODE TABLE DetectionMethod(
        name STRING,
        method_type STRING,
        description STRING,
        PRIMARY KEY(name)
    )''',
    '''CREATE NODE TABLE Indicator(
        name STRING,
        category STRING,
        description STRING,
        PRIMARY KEY(name)
    )''',
    '''CREATE NODE TABLE DataSource(
        name STRING,
        freshness STRING,
        description STRING,
        PRIMARY KEY(name)
    )''',
]

for statement in node_table_ddl:
    display(Markdown(f'```cypher\n{statement.strip()}\n```'))

## 3. Write the Kuzu `CREATE REL TABLE` statements

A relationship table declares which node table can appear on the left, which node table can appear on the right, and which properties live on the edge.

The direction matters. `Detects` is defined as `DetectionMethod -> FraudType`, so this pattern is valid:

```cypher
(method:DetectionMethod)-[:Detects]->(fraud:FraudType)
```

The reverse direction is not the schema we are teaching here.

In [ ]:
relationship_table_ddl = [
    '''CREATE REL TABLE Detects(
        FROM DetectionMethod TO FraudType,
        confidence DOUBLE
    )''',
    '''CREATE REL TABLE Uses(
        FROM DetectionMethod TO Indicator,
        weight DOUBLE
    )''',
    '''CREATE REL TABLE Analyzes(
        FROM DetectionMethod TO DataSource,
        priority INT64
    )''',
]

for statement in relationship_table_ddl:
    display(Markdown(f'```cypher\n{statement.strip()}\n```'))

## 4. Create a fresh Kuzu database

Kuzu is embedded: the database lives in a local folder and the Python process opens it directly. No separate server is required.

This lab uses `output/fraud_manual_schema.kuzu` so it does not overwrite the database created by the other fraud notebook.

In [ ]:
if MANUAL_DB.exists():
    if MANUAL_DB.is_dir():
        shutil.rmtree(MANUAL_DB)
    else:
        MANUAL_DB.unlink()

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
database = kuzu.Database(str(MANUAL_DB))
connection = kuzu.Connection(database)

display(Markdown(f'Created fresh Kuzu database at `{MANUAL_DB}`.'))

## 5. Execute the schema statements

Create node tables first. Relationship tables refer to node tables, so Kuzu needs the node tables to exist before relationships can be defined.

In [ ]:
for statement in node_table_ddl:
    connection.execute(statement)

for statement in relationship_table_ddl:
    connection.execute(statement)

display(Markdown('Created **4 node tables** and **3 relationship tables**.'))

## 6. Insert nodes from `data/fraud.json`

The JSON node rows become Cypher `CREATE` statements. A row like this:

```json
{
  "name": "Payment Fraud",
  "severity": "Medium",
  "description": "Unauthorized or manipulated payment"
}
```

becomes this kind of Cypher:

```cypher
CREATE (n:FraudType {name: 'Payment Fraud', severity: 'Medium', description: 'Unauthorized or manipulated payment'})
```

In [ ]:
def create_node_statement(label, row):
    return f'CREATE (n:{label} {property_map(row)})'

sample_node_statement = create_node_statement('FraudType', dataset['nodes']['FraudType'][2])
display(Markdown(f'```cypher\n{sample_node_statement}\n```'))

In [ ]:
node_insert_count = 0
for label, rows in dataset['nodes'].items():
    for row in rows:
        connection.execute(create_node_statement(label, row))
        node_insert_count += 1

display(Markdown(f'Inserted **{node_insert_count} nodes**.'))

Check the node counts before creating relationships.

In [ ]:
node_counts = []
for node in spec.nodes:
    table = execute(connection, f'MATCH (n:{node.name}) RETURN COUNT(n) AS count')
    node_counts.append({'Node table': node.name, 'Rows': table.rows[0][0]})

display(HTML(records_to_html(node_counts, ('Node table', 'Rows'))))

## 7. Create relationships by matching primary keys

A relationship row in JSON contains `from`, `to`, and relationship properties:

```json
{
  "from": "Machine Learning",
  "to": "Payment Fraud",
  "properties": {"confidence": 0.89}
}
```

To create that edge in Kuzu, first match the source and target nodes by primary key, then create the relationship:

```cypher
MATCH (source:DetectionMethod {name: 'Machine Learning'}),
      (target:FraudType {name: 'Payment Fraud'})
CREATE (source)-[:Detects {confidence: 0.89}]->(target)
```

In [ ]:
def create_relationship_statement(rel_name, row):
    rel_spec = spec.relationship_map[rel_name]
    from_spec = spec.node_map[rel_spec.from_node]
    to_spec = spec.node_map[rel_spec.to_node]
    properties = row.get('properties', {})
    rel_properties = f' {property_map(properties)}' if properties else ''
    return (
        f'MATCH (source:{rel_spec.from_node} '
        f'{{{from_spec.primary_key}: {cypher_literal(row["from"])}}}), '
        f'(target:{rel_spec.to_node} '
        f'{{{to_spec.primary_key}: {cypher_literal(row["to"])}}}) '
        f'CREATE (source)-[:{rel_name}{rel_properties}]->(target)'
    )

sample_relationship_statement = create_relationship_statement('Detects', dataset['relationships']['Detects'][0])
display(Markdown(f'```cypher\n{sample_relationship_statement}\n```'))

In [ ]:
relationship_insert_count = 0
for rel_name, rows in dataset['relationships'].items():
    for row in rows:
        connection.execute(create_relationship_statement(rel_name, row))
        relationship_insert_count += 1

display(Markdown(f'Inserted **{relationship_insert_count} relationships**.'))

In [ ]:
relationship_counts = []
for rel in spec.relationships:
    cypher = f'MATCH (a:{rel.from_node})-[r:{rel.name}]->(b:{rel.to_node}) RETURN COUNT(r) AS count'
    table = execute(connection, cypher)
    relationship_counts.append({'Relationship table': rel.name, 'Rows': table.rows[0][0]})

display(HTML(records_to_html(relationship_counts, ('Relationship table', 'Rows'))))

## 8. Query the graph

Now the tables and relationships exist, so students can traverse the graph with Cypher.

In [ ]:
def show_query(title, cypher):
    display(Markdown(f'### {title}'))
    display(Markdown(f'```cypher\n{cypher.strip()}\n```'))
    display(HTML(table_to_html(execute(connection, cypher))))

show_query('Methods that detect fraud types', '''
MATCH (m:DetectionMethod)-[r:Detects]->(f:FraudType)
RETURN m.name AS method, f.name AS fraud_type, f.severity AS severity, r.confidence AS confidence
ORDER BY confidence DESC
''')

In [ ]:
show_query('Indicators used by detection methods', '''
MATCH (m:DetectionMethod)-[r:Uses]->(i:Indicator)
RETURN m.name AS method, i.name AS indicator, i.category AS category, r.weight AS weight
ORDER BY method, weight DESC
''')

In [ ]:
show_query('Full fraud-detection pipeline', '''
MATCH (d:DataSource)<-[:Analyzes]-(m:DetectionMethod)-[:Uses]->(i:Indicator),
      (m)-[r:Detects]->(f:FraudType)
RETURN d.name AS data_source, m.name AS method, i.name AS indicator, f.name AS fraud_type, r.confidence AS confidence
ORDER BY method, confidence DESC
''')

## 9. Show that relationship direction matters

This query follows the valid direction from `DetectionMethod` to `Indicator`.

In [ ]:
show_query('Valid direction: method uses indicator', '''
MATCH (m:DetectionMethod)-[:Uses]->(i:Indicator)
RETURN m.name AS method, i.name AS indicator
ORDER BY method, indicator
''')

For class discussion: the sentence `Indicator uses DetectionMethod` does not match the business meaning. The schema should encode the relationship direction that makes the business sentence true.

## 10. Visualize the graph

The diagram below is generated from the same fraud dataset, so students can compare the visual graph to the tables and Cypher patterns they just created.

In [ ]:
display(SVG(fraud_graph_svg()))

## 11. Meaningful outcome query

To make the graph useful, turn traversal results into a decision. This query ranks fraud types by best confidence and method coverage.

In [ ]:
show_query('Which fraud type should analysts review first?', '''
MATCH (m:DetectionMethod)-[r:Detects]->(f:FraudType)
WITH f, COUNT(m) AS detecting_methods, MAX(r.confidence) AS best_confidence
RETURN f.name AS fraud_type, f.severity AS severity, detecting_methods, best_confidence
ORDER BY best_confidence DESC, detecting_methods DESC
''')

## 12. Student challenge

Ask students to make one safe change:

1. Add an `Indicator` named `Impossible Travel` to `data/fraud.json`.
2. Add a `Uses` relationship from `Device Intelligence` to `Impossible Travel` with weight `0.88`.
3. Rerun this notebook from the top.
4. Confirm the node count, relationship count, visualization, and indicator query changed.

Reflection question: why does the relationship row only need `from`, `to`, and `properties` instead of repeating the whole source and target nodes?